In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType , IntegerType, StringType ,StructField

In [3]:
spark = SparkSession.builder\
.appName("spark-ops")\
.enableHiveSupport()\
.getOrCreate()

26/06/08 03:06:02 INFO SparkEnv: Registering MapOutputTracker
26/06/08 03:06:02 INFO SparkEnv: Registering BlockManagerMaster
26/06/08 03:06:02 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/06/08 03:06:02 INFO SparkEnv: Registering OutputCommitCoordinator


In [4]:
!hadoop fs -ls /data


Found 6 items
-rw-r--r--   2 root hadoop    1058478 2026-06-04 17:20 /data/customers.csv
-rw-r--r--   2 root hadoop   26214400 2026-06-04 17:48 /data/customers_500.csv
-rw-r--r--   2 root hadoop        209 2026-06-07 07:42 /data/dates_data.csv
-rw-r--r--   2 root hadoop     653137 2026-06-06 16:16 /data/messy_customer_data.csv
drwxr-xr-x   - root hadoop          0 2026-06-06 16:55 /data/write_output.csv
drwxr-xr-x   - root hadoop          0 2026-06-06 17:00 /data/write_output1.csv


In [6]:
df = spark.read\
.format('csv')\
.option('header','true')\
.option('inferSchema','true')\
.load('/data/customers.csv')

In [9]:
df.show()

+-----------+-----------+---------+-----------+-------+-----------------+---------+
|customer_id|       name|     city|      state|country|registration_date|is_active|
+-----------+-----------+---------+-----------+-------+-----------------+---------+
|          0| Customer_0|     Pune|Maharashtra|  India|       2023-05-17|     true|
|          1| Customer_1|Bangalore| Tamil Nadu|  India|       2023-11-29|     true|
|          2| Customer_2|Hyderabad|Maharashtra|  India|       2023-02-08|     true|
|          3| Customer_3|Bangalore|Maharashtra|  India|       2023-03-28|    false|
|          4| Customer_4|Ahmedabad|  Karnataka|  India|       2023-06-04|    false|
|          5| Customer_5|Hyderabad| Tamil Nadu|  India|       2023-06-09|     true|
|          6| Customer_6|     Pune|  Telangana|  India|       2023-03-05|     true|
|          7| Customer_7|Ahmedabad|  Karnataka|  India|       2023-08-11|    false|
|          8| Customer_8|     Pune|  Karnataka|  India|       2023-12-28|   

In [10]:
spark.sql('show tables').show()

ivysettings.xml file not found in HIVE_HOME or HIVE_CONF_DIR,/etc/hive/conf.dist/ivysettings.xml will be used


+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



### Types of SQL tables:
    1. Temporary Table: Exists only for the duration of the session . Schema and data are stored in memory.
    2. Global Temporary Table: Access across all the sessions of same application.
    3. Persistent Table: Stored in metastore(e.g., Hive),with data saved on disk(HDFS/S3).Accessible across sessions.

#### Temporary 

In [11]:
df.createOrReplaceTempView("temp_customers")

In [12]:
spark.sql('show tables').show()

+---------+--------------+-----------+
|namespace|     tableName|isTemporary|
+---------+--------------+-----------+
|         |temp_customers|       true|
+---------+--------------+-----------+



In [14]:
spark.sql('select * from temp_customers').show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-05-17|     true|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-11-29|     true|
|          2|Customer_2|Hyderabad|Maharashtra|  India|       2023-02-08|     true|
|          3|Customer_3|Bangalore|Maharashtra|  India|       2023-03-28|    false|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-06-04|    false|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows



In [19]:
spark_new = spark.newSession()

In [20]:
spark_new.sql('show tables').show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [22]:
spark_new.sql('select * from temp_customers').show(5)

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `temp_customers` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 14;
'Project [*]
+- 'UnresolvedRelation [temp_customers], [], false


##### this was temporary table, so its only accesed on previous session "spark" not on "spark_new".

#### Global Temporary Table

In [25]:
df.createOrReplaceGlobalTempView('globalCustomers')

In [28]:
spark.sql('show tables in global_temp ').show()

+-----------+---------------+-----------+
|  namespace|      tableName|isTemporary|
+-----------+---------------+-----------+
|global_temp|globalcustomers|       true|
|           | temp_customers|       true|
+-----------+---------------+-----------+



In [30]:
 spark_new.sql('select * from global_temp.globalCustomers').show(5) 
##here we took different session "spark_new". we created table in "spark" session.

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-05-17|     true|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-11-29|     true|
|          2|Customer_2|Hyderabad|Maharashtra|  India|       2023-02-08|     true|
|          3|Customer_3|Bangalore|Maharashtra|  India|       2023-03-28|    false|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-06-04|    false|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows



#### Persistent Table

In [32]:
df.write.mode('overwrite').saveAsTable("persistentCustomer")

26/06/08 03:27:53 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


In [33]:
spark.sql('show tables').show()

+---------+------------------+-----------+
|namespace|         tableName|isTemporary|
+---------+------------------+-----------+
|  default|persistentcustomer|      false|
|         |    temp_customers|       true|
+---------+------------------+-----------+

